# Release16 Drawdown-Drift + Trend Reward/Checkpoint Screen v1

This notebook turns the failed `release16_repaired_bc_warm_start_official_replay_v1` handoff into a pre-training reward/checkpoint screen.

## Question

Can the failed release16 repaired-BC official warm-start replay produce a concrete reward/checkpoint objective that is safe to send into a short fold-local BC/PPO training screen, while preserving the hard rule that no actor or runtime overlay is promoted from this diagnostic?

## Baselines and role

- Canonical baseline: `soxlbacktest_policy_reimplementation_v1` / qlib-converted SOXLBacktest reference.
- Direct comparison: `release16_repaired_bc_warm_start_official_replay_v1`.
- Closest checkpoint reference: selected by reward/checkpoint loss from the official warm-start candidates.
- Role: `diagnose`.
- Changed layer: `reward`, `checkpoint_selection`, `model_selection`.
- Main target: walk-forward `relative_return >= +30pp` and `MaxDD <= 20%`.
- Strict risk context: `MaxDD <= 15%` is recorded but is not the main user-requested gate for this turn.
- Accepted trade-off: the notebook may allow only a short fold-local training screen. It cannot promote an actor, retained checkpoint, runtime overlay, or factor.

## Historical context

The source replay repaired supervised BC slice fit, but official warm-start replay still failed every selected fold at the 20% MaxDD gate. The next valid mechanism from that card was to repair the objective with drawdown-drift plus trend participation before PPO.


In [1]:
from __future__ import annotations

import json
import sys
from dataclasses import MISSING, fields
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "soxl_quant").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.drawdown_drift_reward_screen import (
    DrawdownDriftRewardScreenConfig,
    render_drawdown_drift_reward_screen_review,
    screen_drawdown_drift_reward_recipes,
    summarize_drawdown_drift_reward_screen,
)
from src.soxl_quant.app.reward_checkpoint_screen import (
    RewardCheckpointScreenConfig,
    render_reward_checkpoint_screen_review,
    score_reward_checkpoint_candidates,
    summarize_reward_checkpoint_screen,
)
from src.soxl_quant.config import RewardConfig

EXPERIMENT = "release16_drawdown_drift_trend_reward_checkpoint_screen_v1"
CARD_DIR = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
SOURCE_REPLAY = ROOT / "runtime" / "model_selection" / "card" / "release16_repaired_bc_warm_start_official_replay_v1"
SOXLBACKTEST_BASELINE = ROOT / "runtime" / "model_selection" / "card" / "soxlbacktest_qlib_walkforward_baseline_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
STRICT_CONTEXT_MAX_DD_PCT = 15.0
SOURCE_SELECTED_CANDIDATE = "release128_cb995_obj000"


## Inputs

Use the official warm-start replay paths and metrics from the failed release16 repaired-BC card. The SOXLBacktest qlib conversion is loaded only as a reference baseline surface; it is not treated as a promoted model.


In [2]:
metrics = pd.read_csv(SOURCE_REPLAY / "warm_start_official_metrics.csv")
manifest = pd.read_csv(SOURCE_REPLAY / "daily_manifest.csv")
drawdown_intervals = pd.read_csv(SOURCE_REPLAY / "drawdown_intervals.csv")
candidate_summary = pd.read_csv(SOURCE_REPLAY / "candidate_summary.csv")
source_summary = json.loads((SOURCE_REPLAY / "summary.json").read_text(encoding="utf-8"))
source_selection = json.loads((SOURCE_REPLAY / "selection.json").read_text(encoding="utf-8"))
soxlbacktest_metrics = pd.read_csv(SOXLBACKTEST_BASELINE / "candidate_metrics.csv")

input_overview = pd.DataFrame(
    [
        {"input": "warm_start_official_metrics", "rows": len(metrics), "path": str((SOURCE_REPLAY / "warm_start_official_metrics.csv").relative_to(ROOT))},
        {"input": "daily_manifest", "rows": len(manifest), "path": str((SOURCE_REPLAY / "daily_manifest.csv").relative_to(ROOT))},
        {"input": "drawdown_intervals", "rows": len(drawdown_intervals), "path": str((SOURCE_REPLAY / "drawdown_intervals.csv").relative_to(ROOT))},
        {"input": "candidate_summary", "rows": len(candidate_summary), "path": str((SOURCE_REPLAY / "candidate_summary.csv").relative_to(ROOT))},
        {"input": "soxlbacktest_candidate_metrics", "rows": len(soxlbacktest_metrics), "path": str((SOXLBACKTEST_BASELINE / "candidate_metrics.csv").relative_to(ROOT))},
    ]
)
input_overview


,input,rows,path
0,warm_start_official_metrics,9,runtime/model_selection/card/release16_repaire...
1,daily_manifest,9,runtime/model_selection/card/release16_repaire...
2,drawdown_intervals,9,runtime/model_selection/card/release16_repaire...
3,candidate_summary,3,runtime/model_selection/card/release16_repaire...
4,soxlbacktest_candidate_metrics,24,runtime/model_selection/card/soxlbacktest_qlib...


In [3]:
display(
    metrics[
        [
            "candidate",
            "role",
            "fold",
            "official_warm_relative_return",
            "official_warm_max_dd",
            "official_warm_avg_pos",
            "full_gate_20dd",
            "strict_gate_15dd",
        ]
    ].sort_values(["candidate", "fold"])
)

baseline_context = soxlbacktest_metrics[
    soxlbacktest_metrics["window"].isin(["wf_2020", "wf_2022", "wf_2023_2024", "ytd_2026"])
][
    [
        "candidate",
        "role",
        "window",
        "rulebase_return_pct",
        "rulebase_max_drawdown_pct",
        "average_position_pct",
        "trade_count",
        "qlib_calendar_aligned",
    ]
].copy()

baseline_context.head(24)


,candidate,role,fold,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,full_gate_20dd,strict_gate_15dd
0,current_readiness_baseline,direct_baseline,wf_2020,-41.596766,34.734407,0.402347,False,False
1,current_readiness_baseline,direct_baseline,wf_2022,65.866980,24.224438,0.092157,False,False
2,current_readiness_baseline,direct_baseline,wf_2023_2024,24.861597,13.917934,0.243178,False,False
3,release128_cb995_obj000,selected_repaired_bc,wf_2020,-25.013978,34.340318,0.384665,False,False
4,release128_cb995_obj000,selected_repaired_bc,wf_2022,62.347356,26.021628,0.110629,False,False
5,release128_cb995_obj000,selected_repaired_bc,wf_2023_2024,26.481179,16.730406,0.243529,False,False
6,release64_cb99_obj020,prior_official_reference,wf_2020,-2.028796,26.706749,0.344693,False,False
7,release64_cb99_obj020,prior_official_reference,wf_2022,64.650534,24.625320,0.116979,False,False
8,release64_cb99_obj020,prior_official_reference,wf_2023_2024,34.581169,18.964834,0.284346,True,False


,candidate,role,window,rulebase_return_pct,rulebase_max_drawdown_pct,average_position_pct,trade_count,qlib_calendar_aligned
0,legacy_default,buy_sell_reference,wf_2020,220.777205,54.275144,79.102332,4,True
1,legacy_default,buy_sell_reference,wf_2022,-79.396371,84.918053,86.692988,8,True
2,legacy_default,buy_sell_reference,wf_2023_2024,2.935118,32.461810,31.043492,4,True
3,legacy_default,buy_sell_reference,ytd_2026,132.077977,13.554089,16.220734,3,True
4,risk_cap_35_15_05,medium_risk_probe,wf_2020,69.604552,17.942350,31.031976,4,True
5,risk_cap_35_15_05,medium_risk_probe,wf_2022,-37.953340,43.271216,27.872112,10,True
6,risk_cap_35_15_05,medium_risk_probe,wf_2023_2024,4.407749,2.628936,2.790317,5,True
7,risk_cap_35_15_05,medium_risk_probe,ytd_2026,18.308910,0.777618,1.785046,3,True
8,low_risk_20_08_02,near_drawdown_boundary_probe,wf_2020,35.609710,9.073791,17.395810,5,True
9,low_risk_20_08_02,near_drawdown_boundary_probe,wf_2022,-21.715680,24.732060,13.705139,11,True


## Gate-Gap Attribution

The 20% MaxDD gate is the main screen for this user-requested loop. A strict 15% attribution is also written as risk context because previous release16 work used it as a stricter promotion-risk threshold.


In [4]:
def classify_failure(relative_gap: float, max_dd_gap: float) -> str:
    if relative_gap <= 0.0 and max_dd_gap <= 0.0:
        return "pass"
    if relative_gap > 0.0 and max_dd_gap > 0.0:
        return "joint_tradeoff"
    if max_dd_gap > 0.0:
        return "drawdown_control"
    if relative_gap > 0.0:
        return "underparticipation"
    return "unknown"


def build_gate_gaps(target_max_dd_pct: float) -> pd.DataFrame:
    rows = []
    for row in metrics.to_dict("records"):
        relative_return = float(row["official_warm_relative_return"])
        max_dd = float(row["official_warm_max_dd"])
        relative_gap = max(0.0, TARGET_RELATIVE_RETURN_PP - relative_return)
        max_dd_gap = max(0.0, max_dd - float(target_max_dd_pct))
        gate_pass = relative_gap <= 0.0 and max_dd_gap <= 0.0
        rows.append(
            {
                "profile": str(row["candidate"]),
                "candidate_role": str(row["role"]),
                "window": str(row["fold"]),
                "relative_return": relative_return,
                "max_dd": max_dd,
                "relative_gap": relative_gap,
                "max_dd_gap": max_dd_gap,
                "gate_pass": bool(gate_pass),
                "failure_type": classify_failure(relative_gap, max_dd_gap),
                "avg_pos": float(row["official_warm_avg_pos"]),
                "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
                "target_max_dd_pct": float(target_max_dd_pct),
                "daily_path": str(row["daily_path"]),
            }
        )
    return pd.DataFrame(rows).sort_values(["profile", "window"]).reset_index(drop=True)


gate_gaps = build_gate_gaps(TARGET_MAX_DD_PCT)
strict_gate_gaps_15dd = build_gate_gaps(STRICT_CONTEXT_MAX_DD_PCT)

gate_gaps


,profile,candidate_role,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type,avg_pos,target_relative_return_pp,target_max_dd_pct,daily_path
0,current_readiness_baseline,direct_baseline,wf_2020,-41.596766,34.734407,71.596766,14.734407,False,joint_tradeoff,0.402347,30.0,20.0,runtime/model_selection/card/release16_repaire...
1,current_readiness_baseline,direct_baseline,wf_2022,65.866980,24.224438,0.000000,4.224438,False,drawdown_control,0.092157,30.0,20.0,runtime/model_selection/card/release16_repaire...
2,current_readiness_baseline,direct_baseline,wf_2023_2024,24.861597,13.917934,5.138403,0.000000,False,underparticipation,0.243178,30.0,20.0,runtime/model_selection/card/release16_repaire...
3,release128_cb995_obj000,selected_repaired_bc,wf_2020,-25.013978,34.340318,55.013978,14.340318,False,joint_tradeoff,0.384665,30.0,20.0,runtime/model_selection/card/release16_repaire...
4,release128_cb995_obj000,selected_repaired_bc,wf_2022,62.347356,26.021628,0.000000,6.021628,False,drawdown_control,0.110629,30.0,20.0,runtime/model_selection/card/release16_repaire...
5,release128_cb995_obj000,selected_repaired_bc,wf_2023_2024,26.481179,16.730406,3.518821,0.000000,False,underparticipation,0.243529,30.0,20.0,runtime/model_selection/card/release16_repaire...
6,release64_cb99_obj020,prior_official_reference,wf_2020,-2.028796,26.706749,32.028796,6.706749,False,joint_tradeoff,0.344693,30.0,20.0,runtime/model_selection/card/release16_repaire...
7,release64_cb99_obj020,prior_official_reference,wf_2022,64.650534,24.625320,0.000000,4.625320,False,drawdown_control,0.116979,30.0,20.0,runtime/model_selection/card/release16_repaire...
8,release64_cb99_obj020,prior_official_reference,wf_2023_2024,34.581169,18.964834,0.000000,0.000000,True,pass,0.284346,30.0,20.0,runtime/model_selection/card/release16_repaire...


In [5]:
strict_gate_gaps_15dd[["profile", "window", "relative_return", "max_dd", "relative_gap", "max_dd_gap", "gate_pass", "failure_type"]]


,profile,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type
0,current_readiness_baseline,wf_2020,-41.596766,34.734407,71.596766,19.734407,False,joint_tradeoff
1,current_readiness_baseline,wf_2022,65.866980,24.224438,0.000000,9.224438,False,drawdown_control
2,current_readiness_baseline,wf_2023_2024,24.861597,13.917934,5.138403,0.000000,False,underparticipation
3,release128_cb995_obj000,wf_2020,-25.013978,34.340318,55.013978,19.340318,False,joint_tradeoff
4,release128_cb995_obj000,wf_2022,62.347356,26.021628,0.000000,11.021628,False,drawdown_control
5,release128_cb995_obj000,wf_2023_2024,26.481179,16.730406,3.518821,1.730406,False,joint_tradeoff
6,release64_cb99_obj020,wf_2020,-2.028796,26.706749,32.028796,11.706749,False,joint_tradeoff
7,release64_cb99_obj020,wf_2022,64.650534,24.625320,0.000000,9.625320,False,drawdown_control
8,release64_cb99_obj020,wf_2023_2024,34.581169,18.964834,0.000000,3.964834,False,drawdown_control


## Reward/Checkpoint Screen

This screen ranks the official replay candidates by gate gaps and drift versus the prior official reference. A lowest-loss candidate is only a diagnostic reference unless every fold passes.


In [6]:
selected_failure_rows = gate_gaps[
    gate_gaps["profile"].astype(str).eq(SOURCE_SELECTED_CANDIDATE)
    & ~gate_gaps["gate_pass"].astype(bool)
].copy()

poor_window_clusters = pd.DataFrame(
    [
        {
            "cluster_id": f"release16_{row.window}",
            "dominant_failure_type": row.failure_type,
            "window": row.window,
            "source": "release16_repaired_bc_warm_start_official_replay_v1",
        }
        for row in selected_failure_rows.itertuples(index=False)
    ]
)

feature_groups = pd.DataFrame(
    [
        {"feature_group": "trend_momentum", "evidence": "release16 handoff requires trend participation repair"},
        {"feature_group": "relative_strength_spread", "evidence": "underparticipation/trend capture failure"},
        {"feature_group": "drawdown_path", "evidence": "official MaxDD gate failures"},
        {"feature_group": "volatility_regime", "evidence": "official drawdown intervals have stress exposure"},
    ]
)

checkpoint_scores = score_reward_checkpoint_candidates(
    gate_gaps,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_groups,
    reference_candidate="release64_cb99_obj020",
    config=RewardCheckpointScreenConfig(
        target_relative_return=TARGET_RELATIVE_RETURN_PP,
        target_max_dd=TARGET_MAX_DD_PCT,
    ),
)
checkpoint_summary = summarize_reward_checkpoint_screen(checkpoint_scores)
checkpoint_review_md = render_reward_checkpoint_screen_review(checkpoint_scores, checkpoint_summary)

display(checkpoint_scores)
checkpoint_summary


,candidate,window_count,gate_pass_count,full_gate,reward_checkpoint_loss,gate_gap_loss,relative_drift_loss,max_dd_drift_loss,short_volshock_slice_loss,min_relative_return,max_max_dd,failure_mix,dominant_failure_type,required_objective_terms,short_volshock_target_samples,mean_short_volshock_accuracy
0,release64_cb99_obj020,3,1,False,145.349492,145.349492,0.000000,0.000000,0.0,-2.028796,26.706749,"joint_tradeoff:1,drawdown_control:1,pass:1",joint_tradeoff,"[""checkpoint_objective_loss"", ""drawdown_drift_...",0,NaN
1,release128_cb995_obj000,3,0,False,315.648716,262.152250,33.388349,9.029876,0.0,-25.013978,34.340318,"joint_tradeoff:1,drawdown_control:1,underparti...",joint_tradeoff,"[""checkpoint_objective_loss"", ""drawdown_drift_...",0,NaN
2,current_readiness_baseline,3,0,False,318.783784,266.323611,49.287541,8.027657,0.0,-41.596766,34.734407,"joint_tradeoff:1,drawdown_control:1,underparti...",joint_tradeoff,"[""checkpoint_objective_loss"", ""drawdown_drift_...",0,NaN


{'decision': 'reward_checkpoint_screen_no_promotion',
 'selected_candidate': 'release64_cb99_obj020',
 'selected_reward_checkpoint_loss': 145.34949171005215,
 'selected_gate_gap_loss': 145.34949171005215,
 'selected_failure_mix': 'joint_tradeoff:1,drawdown_control:1,pass:1',
 'selected_min_relative_return': -2.028796400955727,
 'selected_max_max_dd': 26.70674924237374,
 'promotion_allowed': False,
 'ppo_allowed_next': False,
 'required_objective_terms': ['checkpoint_objective_loss',
  'drawdown_drift_penalty',
  'trend_preservation_floor'],
 'next_mechanism': 'train_drawdown_drift_reward_or_checkpoint_recipe'}

## Drawdown-Drift + Trend Reward Recipe Screen

The recipe screen is run per candidate. This avoids overfitting the reward recipe to only the supervised-BC-selected candidate when the official checkpoint screen says the older release64 reference is the lowest-loss diagnostic path.


In [7]:
def load_daily_paths(candidate: str) -> dict[str, pd.DataFrame]:
    rows = manifest[manifest["candidate"].astype(str).eq(str(candidate))].copy()
    return {
        f"{row.candidate}__{row.fold}": pd.read_csv(ROOT / row.path)
        for row in rows.itertuples(index=False)
    }

recipe_frames = []
recipe_summaries = []
for candidate in sorted(manifest["candidate"].astype(str).unique()):
    candidate_paths = load_daily_paths(candidate)
    candidate_scores = screen_drawdown_drift_reward_recipes(
        candidate_paths,
        gate_gaps=gate_gaps,
        selected_candidate=candidate,
        config=DrawdownDriftRewardScreenConfig(
            target_relative_return_pp=TARGET_RELATIVE_RETURN_PP,
            target_max_dd_pct=TARGET_MAX_DD_PCT,
        ),
    )
    candidate_scores = candidate_scores.copy()
    candidate_scores.insert(0, "candidate", candidate)
    role_values = metrics[metrics["candidate"].astype(str).eq(candidate)]["role"].astype(str).unique().tolist()
    candidate_scores.insert(1, "candidate_role", role_values[0] if role_values else "")
    recipe_frames.append(candidate_scores)
    candidate_summary_row = summarize_drawdown_drift_reward_screen(candidate_scores.drop(columns=["candidate", "candidate_role"], errors="ignore"))
    candidate_summary_row["candidate"] = candidate
    candidate_summary_row["candidate_role"] = role_values[0] if role_values else ""
    recipe_summaries.append(candidate_summary_row)

recipe_scores = pd.concat(recipe_frames, ignore_index=True).sort_values(
    ["recipe_ready", "reward_recipe_loss", "candidate", "recipe"],
    ascending=[False, True, True, True],
).reset_index(drop=True)
recipe_summary_by_candidate = pd.DataFrame(recipe_summaries).sort_values(
    ["training_allowed_next", "selected_reward_recipe_loss", "candidate"],
    ascending=[False, True, True],
).reset_index(drop=True)

selected_checkpoint_candidate = str(checkpoint_summary.get("selected_candidate", ""))
selected_checkpoint_recipe_frame = recipe_scores[recipe_scores["candidate"].astype(str).eq(selected_checkpoint_candidate)].copy()
selected_repaired_recipe_frame = recipe_scores[recipe_scores["candidate"].astype(str).eq(SOURCE_SELECTED_CANDIDATE)].copy()

recipe_summary = summarize_drawdown_drift_reward_screen(
    selected_checkpoint_recipe_frame.drop(columns=["candidate", "candidate_role"], errors="ignore")
)
recipe_summary["selected_candidate"] = selected_checkpoint_candidate
recipe_summary["source_selected_candidate"] = SOURCE_SELECTED_CANDIDATE
recipe_summary["source_selected_candidate_recipe"] = (
    selected_repaired_recipe_frame.iloc[0]["recipe"] if not selected_repaired_recipe_frame.empty else ""
)
recipe_summary["source_selected_candidate_recipe_ready"] = (
    bool(selected_repaired_recipe_frame.iloc[0]["recipe_ready"]) if not selected_repaired_recipe_frame.empty else False
)
recipe_review_md = render_drawdown_drift_reward_screen_review(
    selected_checkpoint_recipe_frame.drop(columns=["candidate", "candidate_role"], errors="ignore"),
    recipe_summary,
    top_n=12,
)

display(recipe_summary_by_candidate)
recipe_scores[[
    "candidate",
    "candidate_role",
    "recipe",
    "recipe_ready",
    "reward_recipe_loss",
    "drawdown_failure_capture",
    "trend_underparticipation_capture",
    "overreach_share",
    "active_rows",
    "drawdown_breach_rows",
    "drawdown_captured_rows",
    "trend_opportunity_rows",
    "trend_captured_rows",
]].head(18)


,decision,selected_recipe,selected_reward_recipe_loss,drawdown_failure_capture,trend_underparticipation_capture,overreach_share,training_allowed_next,promotion_allowed,reward_objective_patch,next_mechanism,candidate,candidate_role
0,drawdown_drift_reward_recipe_ready_for_short_t...,dd1.5_thr18_pos0.15_tf1_floor0.25_lb21,1.305765,0.680000,1.0,0.0,True,False,"{'drawdown_drift_penalty': 1.5, 'drawdown_drif...",short_foldlocal_bc_or_ppo_training_screen,release64_cb99_obj020,prior_official_reference
1,drawdown_drift_reward_recipe_ready_for_short_t...,dd1.5_thr18_pos0.15_tf1_floor0.25_lb21,2.555440,0.536424,1.0,0.0,True,False,"{'drawdown_drift_penalty': 1.5, 'drawdown_drif...",short_foldlocal_bc_or_ppo_training_screen,release128_cb995_obj000,selected_repaired_bc
2,drawdown_drift_reward_recipe_ready_for_short_t...,dd1.5_thr18_pos0.15_tf1_floor0.25_lb21,2.649655,0.563636,1.0,0.0,True,False,"{'drawdown_drift_penalty': 1.5, 'drawdown_drif...",short_foldlocal_bc_or_ppo_training_screen,current_readiness_baseline,direct_baseline


,candidate,candidate_role,recipe,recipe_ready,reward_recipe_loss,drawdown_failure_capture,trend_underparticipation_capture,overreach_share,active_rows,drawdown_breach_rows,drawdown_captured_rows,trend_opportunity_rows,trend_captured_rows
0,release64_cb99_obj020,prior_official_reference,dd1.5_thr18_pos0.15_tf1_floor0.25_lb21,True,1.305765,0.68,1.0,0.0,107,50,34,42,42
1,release64_cb99_obj020,prior_official_reference,dd2_thr18_pos0.15_tf1_floor0.25_lb21,True,1.399801,0.68,1.0,0.0,107,50,34,42,42
2,release64_cb99_obj020,prior_official_reference,dd1.5_thr18_pos0.1_tf1_floor0.25_lb21,True,1.479870,0.78,1.0,0.0,112,50,39,42,42
3,release64_cb99_obj020,prior_official_reference,dd3_thr18_pos0.15_tf1_floor0.25_lb21,True,1.587875,0.68,1.0,0.0,107,50,34,42,42
4,release64_cb99_obj020,prior_official_reference,dd1.5_thr18_pos0.05_tf1_floor0.25_lb21,True,1.603530,1.00,1.0,0.0,123,50,50,42,42
5,release64_cb99_obj020,prior_official_reference,dd2_thr18_pos0.1_tf1_floor0.25_lb21,True,1.647175,0.78,1.0,0.0,112,50,39,42,42
6,release64_cb99_obj020,prior_official_reference,dd1.5_thr15_pos0.15_tf1_floor0.25_lb21,True,1.754948,0.68,1.0,0.0,107,50,34,42,42
7,release64_cb99_obj020,prior_official_reference,dd2_thr18_pos0.05_tf1_floor0.25_lb21,True,1.841208,1.00,1.0,0.0,123,50,50,42,42
8,release64_cb99_obj020,prior_official_reference,dd1.5_thr18_pos0.15_tf1_floor0.35_lb21,True,1.869113,0.68,1.0,0.0,137,50,34,62,62
9,release64_cb99_obj020,prior_official_reference,dd2_thr18_pos0.15_tf1_floor0.35_lb21,True,1.942558,0.68,1.0,0.0,137,50,34,62,62


In [8]:
recipe_summary


{'decision': 'drawdown_drift_reward_recipe_ready_for_short_training_screen',
 'selected_recipe': 'dd1.5_thr18_pos0.15_tf1_floor0.25_lb21',
 'selected_reward_recipe_loss': 1.3057646575311084,
 'drawdown_failure_capture': 0.68,
 'trend_underparticipation_capture': 1.0,
 'overreach_share': 0.0,
 'training_allowed_next': True,
 'promotion_allowed': False,
 'reward_objective_patch': {'drawdown_drift_penalty': 1.5,
  'drawdown_drift_position_threshold': 0.15,
  'drawdown_drift_reward_enabled': True,
  'drawdown_drift_threshold_pct': 18.0,
  'trend_preservation_floor_enabled': True,
  'trend_preservation_floor_penalty': 1.0,
  'trend_preservation_floor_position': 0.25,
  'trend_preservation_lookback': 21,
  'trend_preservation_min_buyhold_return_pct': 5.0},
 'next_mechanism': 'short_foldlocal_bc_or_ppo_training_screen',
 'selected_candidate': 'release64_cb99_obj020',
 'source_selected_candidate': 'release128_cb995_obj000',
 'source_selected_candidate_recipe': 'dd1.5_thr18_pos0.15_tf1_floor0.2

## Reward Patch Contract Audit

A ready recipe can only enter a short training screen if every patch key is accepted by `RewardConfig` and consumed by `ContinuousReward`. This prevents sidecar JSON from pretending to be a real training objective.


In [9]:
selected_patch = dict(recipe_summary.get("reward_objective_patch", {}))
reward_config_fields = {field.name: field for field in fields(RewardConfig)}
reward_source = (ROOT / "src" / "soxl_quant" / "policy" / "reward.py").read_text(encoding="utf-8")

base_reward_kwargs = {}
for field in fields(RewardConfig):
    if field.default is not MISSING:
        base_reward_kwargs[field.name] = field.default
    elif field.default_factory is not MISSING:  # type: ignore[attr-defined]
        base_reward_kwargs[field.name] = field.default_factory()  # type: ignore[misc]
    else:
        base_reward_kwargs[field.name] = 0.0
base_reward_kwargs.update(selected_patch)
reward_config_instance = RewardConfig(**base_reward_kwargs)

patch_audit = pd.DataFrame(
    [
        {
            "patch_key": key,
            "patch_value": value,
            "accepted_by_reward_config": key in reward_config_fields,
            "consumed_by_continuous_reward": key in reward_source,
            "default_value": getattr(RewardConfig, key, None),
        }
        for key, value in selected_patch.items()
    ]
).sort_values("patch_key").reset_index(drop=True)

reward_patch_contract_gate = bool(
    not patch_audit.empty
    and patch_audit["accepted_by_reward_config"].all()
    and patch_audit["consumed_by_continuous_reward"].all()
    and isinstance(reward_config_instance, RewardConfig)
)

patch_contract_summary = {
    "reward_patch_contract_gate": reward_patch_contract_gate,
    "accepted_key_count": int(patch_audit["accepted_by_reward_config"].sum()) if not patch_audit.empty else 0,
    "consumed_key_count": int(patch_audit["consumed_by_continuous_reward"].sum()) if not patch_audit.empty else 0,
    "patch_key_count": int(len(patch_audit)),
    "reward_config_instantiated": isinstance(reward_config_instance, RewardConfig),
    "focused_tests": [
        "tests/test_reward.py::test_drawdown_drift_reward_penalizes_position_after_drawdown_threshold",
        "tests/test_reward.py::test_trend_preservation_floor_penalizes_underposition_only_outside_drawdown_breach",
        "tests/test_drawdown_drift_reward_screen.py",
        "tests/test_reward_checkpoint_screen.py",
    ],
}

display(patch_audit)
patch_contract_summary


,patch_key,patch_value,accepted_by_reward_config,consumed_by_continuous_reward,default_value
0,drawdown_drift_penalty,1.5,True,True,0.0
1,drawdown_drift_position_threshold,0.15,True,True,0.05
2,drawdown_drift_reward_enabled,True,True,True,False
3,drawdown_drift_threshold_pct,18.0,True,True,15.0
4,trend_preservation_floor_enabled,True,True,True,False
5,trend_preservation_floor_penalty,1.0,True,True,0.0
6,trend_preservation_floor_position,0.25,True,True,0.25
7,trend_preservation_lookback,21,True,True,63
8,trend_preservation_min_buyhold_return_pct,5.0,True,True,5.0


{'reward_patch_contract_gate': True,
 'accepted_key_count': 9,
 'consumed_key_count': 9,
 'patch_key_count': 9,
 'reward_config_instantiated': True,
 'focused_tests': ['tests/test_reward.py::test_drawdown_drift_reward_penalizes_position_after_drawdown_threshold',
  'tests/test_reward.py::test_trend_preservation_floor_penalizes_underposition_only_outside_drawdown_breach',
  'tests/test_drawdown_drift_reward_screen.py',
  'tests/test_reward_checkpoint_screen.py']}

## Write Sidecars

The notebook is the primary conclusion. Sidecars are written so the next short training screen can consume the gate attribution, recipe, and patch contract directly.


In [10]:
def clean_for_json(value):
    if isinstance(value, dict):
        return {str(k): clean_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [clean_for_json(v) for v in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value

created_at = datetime.now(timezone.utc).isoformat()
training_allowed_next = bool(recipe_summary.get("training_allowed_next", False) and reward_patch_contract_gate)

selection = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "source_replay": str(SOURCE_REPLAY.relative_to(ROOT)),
    "canonical_baseline": source_summary.get("canonical_baseline", "soxlbacktest_policy_reimplementation_v1"),
    "direct_comparison_baseline": source_summary.get("direct_comparison_baseline", []),
    "soxlbacktest_qlib_baseline": str(SOXLBACKTEST_BASELINE.relative_to(ROOT)),
    "role": "diagnose_reward_checkpoint_objective_before_training",
    "changed_layer": ["reward", "checkpoint_selection", "model_selection"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "strict_context_max_dd_pct": STRICT_CONTEXT_MAX_DD_PCT,
    "source_selected_candidate": SOURCE_SELECTED_CANDIDATE,
    "checkpoint_reference_candidate": checkpoint_summary.get("selected_candidate", ""),
    "checkpoint_screen_decision": checkpoint_summary.get("decision", ""),
    "recipe_decision": recipe_summary.get("decision", ""),
    "selected_recipe_candidate": recipe_summary.get("selected_candidate", ""),
    "selected_recipe": recipe_summary.get("selected_recipe", ""),
    "reward_objective_patch": selected_patch,
    "reward_patch_contract_gate": reward_patch_contract_gate,
    "training_allowed_next": training_allowed_next,
    "training_scope": "short_foldlocal_bc_or_ppo_training_screen_only" if training_allowed_next else "blocked_until_reward_objective_revised",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_mechanism": "short_foldlocal_bc_or_ppo_training_screen" if training_allowed_next else "revise_reward_state_or_checkpoint_objective",
}

summary = {
    **selection,
    "decision": "drawdown_drift_reward_recipe_ready_for_short_training_screen" if training_allowed_next else "drawdown_drift_reward_recipe_no_training_signal",
    "checkpoint_summary": checkpoint_summary,
    "recipe_summary": recipe_summary,
    "patch_contract_summary": patch_contract_summary,
    "source_replay_decision": source_summary.get("decision", ""),
    "source_replay_next_mechanism": source_summary.get("next_mechanism", ""),
    "selected_official_gate_count_20dd": int(
        gate_gaps[gate_gaps["profile"].astype(str).eq(SOURCE_SELECTED_CANDIDATE)]["gate_pass"].sum()
    ),
    "selected_official_fold_count": int(
        gate_gaps[gate_gaps["profile"].astype(str).eq(SOURCE_SELECTED_CANDIDATE)].shape[0]
    ),
    "selected_strict_gate_count_15dd": int(
        strict_gate_gaps_15dd[strict_gate_gaps_15dd["profile"].astype(str).eq(SOURCE_SELECTED_CANDIDATE)]["gate_pass"].sum()
    ),
    "conclusion": [
        "The source selected repaired BC still has 0/3 folds passing the 20% official warm-start gate, so no actor or checkpoint can be promoted.",
        f"The reward/checkpoint screen selects `{checkpoint_summary.get('selected_candidate', '')}` only as the lowest-loss diagnostic reference; its decision remains `{checkpoint_summary.get('decision', '')}`.",
        f"The selected reward recipe `{recipe_summary.get('selected_recipe', '')}` captures {recipe_summary.get('drawdown_failure_capture', 0.0):.1%} of drawdown-breach rows and {recipe_summary.get('trend_underparticipation_capture', 0.0):.1%} of trend-underparticipation rows with overreach_share={recipe_summary.get('overreach_share', 0.0):.1%}.",
        "Every reward patch key is accepted by RewardConfig and consumed by ContinuousReward, so the patch is a real default-off training objective.",
        "Next action is limited to a short fold-local BC/PPO training screen using this reward/checkpoint objective; promotion remains false until official walk-forward replay passes.",
    ],
    "sidecars": [
        "gate_gap_attribution.csv",
        "strict_gate_gap_attribution_15dd.csv",
        "reward_checkpoint_scores.csv",
        "reward_recipe_scores.csv",
        "reward_recipe_summary_by_candidate.csv",
        "reward_objective_patch.json",
        "reward_patch_contract_audit.csv",
        "reward_patch_contract_audit.json",
        "soxlbacktest_reference_context.csv",
        "selection.json",
        "summary.json",
        "reward_checkpoint_review.md",
        "reward_recipe_review.md",
    ],
}

# Sidecars
gate_gaps.to_csv(CARD_DIR / "gate_gap_attribution.csv", index=False)
strict_gate_gaps_15dd.to_csv(CARD_DIR / "strict_gate_gap_attribution_15dd.csv", index=False)
checkpoint_scores.to_csv(CARD_DIR / "reward_checkpoint_scores.csv", index=False)
recipe_scores.to_csv(CARD_DIR / "reward_recipe_scores.csv", index=False)
recipe_summary_by_candidate.to_csv(CARD_DIR / "reward_recipe_summary_by_candidate.csv", index=False)
patch_audit.to_csv(CARD_DIR / "reward_patch_contract_audit.csv", index=False)
baseline_context.to_csv(CARD_DIR / "soxlbacktest_reference_context.csv", index=False)
(CARD_DIR / "reward_objective_patch.json").write_text(json.dumps(clean_for_json(selected_patch), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "reward_patch_contract_audit.json").write_text(json.dumps(clean_for_json(patch_contract_summary), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "selection.json").write_text(json.dumps(clean_for_json(selection), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps(clean_for_json(summary), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "reward_checkpoint_review.md").write_text(checkpoint_review_md, encoding="utf-8")
(CARD_DIR / "reward_recipe_review.md").write_text(recipe_review_md, encoding="utf-8")

summary

{'experiment': 'release16_drawdown_drift_trend_reward_checkpoint_screen_v1',
 'created_at': '2026-06-14T22:08:48.285579+00:00',
 'source_replay': 'runtime/model_selection/card/release16_repaired_bc_warm_start_official_replay_v1',
 'canonical_baseline': 'soxlbacktest_policy_reimplementation_v1',
 'direct_comparison_baseline': ['release16_checkpoint_distillation_bc_repair_screen_v1',
  'release16_release_gate_bc_official_replay_v1'],
 'soxlbacktest_qlib_baseline': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1',
 'role': 'diagnose_reward_checkpoint_objective_before_training',
 'changed_layer': ['reward', 'checkpoint_selection', 'model_selection'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'strict_context_max_dd_pct': 15.0,
 'source_selected_candidate': 'release128_cb995_obj000',
 'checkpoint_reference_candidate': 'release64_cb99_obj020',
 'checkpoint_screen_decision': 'reward_checkpoint_screen_no_promotion',
 'recipe_decision': 'drawdown_dr

## Conclusion

The source repaired-BC official replay is still blocked: the selected source candidate passes `0/3` official warm-start folds at the 20% MaxDD gate and `0/3` at the strict 15% context gate. The checkpoint screen identifies only a lower-loss diagnostic reference, not a promotable actor.

The selected reward recipe is therefore a training-objective handoff, not a runtime improvement claim. It may enter only a short fold-local BC/PPO screen after the patch contract audit passes, and any resulting actor still needs official walk-forward replay before retention or promotion.


In [11]:
display(Markdown("\n".join([
    "### Final Readout",
    f"- decision: `{summary['decision']}`",
    f"- checkpoint_reference_candidate: `{summary['checkpoint_reference_candidate']}`",
    f"- selected_recipe_candidate: `{summary['selected_recipe_candidate']}`",
    f"- selected_recipe: `{summary['selected_recipe']}`",
    f"- training_allowed_next: `{summary['training_allowed_next']}`",
    f"- promotion_allowed: `{summary['promotion_allowed']}`",
    f"- reward_patch_contract_gate: `{summary['reward_patch_contract_gate']}`",
    f"- next_mechanism: `{summary['next_mechanism']}`",
])))

### Final Readout
- decision: `drawdown_drift_reward_recipe_ready_for_short_training_screen`
- checkpoint_reference_candidate: `release64_cb99_obj020`
- selected_recipe_candidate: `release64_cb99_obj020`
- selected_recipe: `dd1.5_thr18_pos0.15_tf1_floor0.25_lb21`
- training_allowed_next: `True`
- promotion_allowed: `False`
- reward_patch_contract_gate: `True`
- next_mechanism: `short_foldlocal_bc_or_ppo_training_screen`